In [1]:
import stim
from IPython.core.display import HTML

In [2]:
def repetition_circuit(distance: int, rounds: int = 1) -> stim.Circuit:
    if distance < 2:
        raise ValueError("Not defined for distance < 2.")

    physical_qubit_count = 2*distance
    data_qubits = list(range(0, physical_qubit_count, 2))
    meas_qubits = list(range(1, physical_qubit_count-1, 2))

    rep_circuit = stim.Circuit()

    for qd in data_qubits:
        rep_circuit.append("QUBIT_COORDS", [qd], [0, qd // 2])
    for qm in meas_qubits:
        rep_circuit.append("QUBIT_COORDS", [qm], [0.5, qm // 2 + 0.5])

    rep_circuit.append("TICK")

    rep_circuit.append("R", data_qubits)

    qec_round = stim.Circuit()
    qec_round.append("TICK")

    # Prepare all measurement qubits
    qec_round.append("R", meas_qubits)

    qec_round.append("TICK")

    # CNOTs from measurement qubits to data qubit above
    for qm in meas_qubits:
        qec_round.append("CNOT", [qm-1, qm])

    qec_round.append("TICK")

    # CNOTs from measurement qubits to data qubit below
    for qm in meas_qubits:
        qec_round.append("CNOT", [qm+1, qm])

    qec_round.append("TICK")

    # Measure syndrome
    qec_round.append("M", meas_qubits)

    qec_round.append("TICK")

    rep_circuit.append(stim.CircuitRepeatBlock(
        repeat_count=rounds,
        body=qec_round,
    ))

    rep_circuit.append("TICK")

    rep_circuit.append("M", data_qubits)

    return rep_circuit

In [3]:
circuit = repetition_circuit(distance=5, rounds=4)
url = circuit.to_crumble_url()
display(HTML(f'<a href="{url}">{url}</a>'))

In [4]:
circuit = stim.Circuit.generated(
    "repetition_code:memory", distance=4, rounds=4, after_clifford_depolarization=0.01
)
url = circuit.to_crumble_url()
display(HTML(f'<a href="{url}">{url}</a>'))
print(url)

https://algassert.com/crumble#circuit=R_0_1_2_3_4_5_6;TICK;CX_0_1_2_3_4_5;DEPOLARIZE2(0.01)0_1_2_3_4_5;TICK;CX_2_1_4_3_6_5;DEPOLARIZE2(0.01)2_1_4_3_6_5;TICK;MR_1_3_5;DT(1,0)rec[-3];DT(3,0)rec[-2];DT(5,0)rec[-1];REPEAT_3_{;TICK;CX_0_1_2_3_4_5;DEPOLARIZE2(0.01)0_1_2_3_4_5;TICK;CX_2_1_4_3_6_5;DEPOLARIZE2(0.01)2_1_4_3_6_5;TICK;MR_1_3_5;SHIFT_COORDS(0,1);DT(1,0)rec[-3]_rec[-6];DT(3,0)rec[-2]_rec[-5];DT(5,0)rec[-1]_rec[-4];};M_0_2_4_6;DT(1,1)rec[-3]_rec[-4]_rec[-7];DT(3,1)rec[-2]_rec[-3]_rec[-6];DT(5,1)rec[-1]_rec[-2]_rec[-5];OI(0)rec[-1]_
